# reduce-gather-sum — ex2: global median across ranks via all_gather + manual sort

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `reduce-gather-sum`. Running the final beacon cell reports progress against the `Distributed: reduce.gather + sum` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Distributed: reduce.gather + sum` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`reduce-gather-sum`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "reduce-gather-sum"
DD_SUBTOPIC = "Distributed: reduce.gather + sum"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## `all_gather` + manual aggregation — quick refresher

`dist.all_gather(gather_list, tensor)` collects each rank's `tensor` into a length-`world_size` list of tensors on EVERY rank. Unlike `gather` (which populates only the dst rank), `all_gather` fans the result out — every rank ends with the same `gather_list`.

**Why all_gather instead of all_reduce.** `all_reduce` collapses to a single aggregate (sum, max, min, product). For aggregations that AREN'T associative-binary — median, percentile, sorted top-k — there's no `ReduceOp` you can pass. You need the per-rank values, on every rank, then compute the aggregation locally.

**Memory trade-off.** `all_gather` holds N tensors on every rank (N×memory). `all_reduce` holds 1 tensor on every rank. For scalar metrics across modest world sizes, the cost is negligible.

**Same gather_list pre-allocation pattern as `gather`.** Callers pre-build the `[t.zeros_like(...) for _ in range(world_size)]` list and pass it as the destination. The collective fills the slots.

### Exercise 2 — global median across ranks via all_gather + manual sort

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Bloom level: Apply
> LO: Apply `dist.all_gather` followed by a local sort + median to compute a global median across ranks — an aggregation `ReduceOp` cannot express.
> Keywords: all_gather, median, manual-aggregation, non-associative
> ```

**KCs targeted:** `all-gather-pre-allocates-gather-list`, `median-from-gathered-tensors`

Implement `ex2_all_gather_median(rank, world_size, dist_module, local_value)`. Compute the global MEDIAN of `world_size` rank-local scalars on every rank.

Steps:
1. Wrap the local value: `tensor = t.tensor([local_value], dtype=t.float32)`.
2. Pre-allocate the gather list: `gather_list = [t.zeros(1, dtype=t.float32) for _ in range(world_size)]`. Build this on EVERY rank (not just rank 0) — `all_gather` populates every rank's list.
3. `dist_module.all_gather(gather_list, tensor)`. After this, every rank's `gather_list[r]` holds rank r's value.
4. Stack into one tensor: `gathered = t.cat(gather_list)` (shape `(world_size,)`).
5. Sort and take the median index — for even `world_size`, average the two middle values:
   ```python
   sorted_vals = t.sort(gathered).values
   mid = world_size // 2
   if world_size % 2 == 1:
       median = sorted_vals[mid].item()
   else:
       median = ((sorted_vals[mid - 1] + sorted_vals[mid]) / 2).item()
   ```
6. Return `median` — a Python float, identical on every rank.

**Why `all_gather`, not `reduce`.** `ReduceOp` only supports associative binary ops (sum, max, min, product). Median requires the FULL distribution to compute — every rank needs the whole set, hence `all_gather`.

In [ ]:
def ex2_all_gather_median(rank: int, world_size: int, dist_module, local_value: float) -> float:
    """Compute global median across ranks via all_gather + sort."""
    raise NotImplementedError()


def _test_ex2():

    import threading
    import types as _types
    import torch as _t_for_fake


    class _FakeReduceOp:
        SUM = 'SUM'
        MAX = 'MAX'
        MIN = 'MIN'
        PRODUCT = 'PROD'


    class _FakeWorld:
        """Shared state across `world_size` rank-threads."""
        def __init__(self, world_size):
            self.world_size = world_size
            self.barrier = threading.Barrier(world_size)
            self.lock = threading.Lock()
            self.scratch = {}
            self.tls = threading.local()
            self.results = [None] * world_size
            # rank-0-only side-effect log (per-call log lines for tests to inspect)
            self.side_effects = []

        def _reduce_op(self, bag, op):
            if op == 'SUM':
                out = bag[0].clone()
                for x in bag[1:]:
                    out = out + x
                return out
            if op == 'MAX':
                out = bag[0].clone()
                for x in bag[1:]:
                    out = _t_for_fake.maximum(out, x)
                return out
            if op == 'MIN':
                out = bag[0].clone()
                for x in bag[1:]:
                    out = _t_for_fake.minimum(out, x)
                return out
            if op == 'PROD':
                out = bag[0].clone()
                for x in bag[1:]:
                    out = out * x
                return out
            raise ValueError(f'unknown fake op {op!r}')

        def all_reduce(self, tensor, op='SUM'):
            rank = self.tls.rank
            self.barrier.wait()
            with self.lock:
                self.scratch.setdefault('ar', [None] * self.world_size)
                self.scratch['ar'][rank] = tensor.detach().clone()
            self.barrier.wait()
            bag = self.scratch['ar']
            reduced = self._reduce_op(bag, op)
            tensor.copy_(reduced)
            self.barrier.wait()
            if rank == 0:
                self.scratch.pop('ar', None)
            self.barrier.wait()

        def reduce(self, tensor, dst, op='SUM'):
            rank = self.tls.rank
            self.barrier.wait()
            with self.lock:
                self.scratch.setdefault('rd', [None] * self.world_size)
                self.scratch['rd'][rank] = tensor.detach().clone()
            self.barrier.wait()
            if rank == dst:
                bag = self.scratch['rd']
                tensor.copy_(self._reduce_op(bag, op))
            self.barrier.wait()
            if rank == 0:
                self.scratch.pop('rd', None)
            self.barrier.wait()

        def broadcast(self, tensor, src):
            rank = self.tls.rank
            self.barrier.wait()
            if rank == src:
                with self.lock:
                    self.scratch['bc'] = tensor.detach().clone()
            self.barrier.wait()
            if rank != src:
                tensor.copy_(self.scratch['bc'])
            self.barrier.wait()
            if rank == 0:
                self.scratch.pop('bc', None)
            self.barrier.wait()

        def gather(self, tensor, gather_list, dst):
            """Mock dist.gather — only dst's gather_list is populated."""
            rank = self.tls.rank
            self.barrier.wait()
            with self.lock:
                self.scratch.setdefault('gth', [None] * self.world_size)
                self.scratch['gth'][rank] = tensor.detach().clone()
            self.barrier.wait()
            if rank == dst:
                bag = self.scratch['gth']
                for i, src_tensor in enumerate(bag):
                    gather_list[i].copy_(src_tensor)
            self.barrier.wait()
            if rank == 0:
                self.scratch.pop('gth', None)
            self.barrier.wait()

        def all_gather(self, gather_list, tensor):
            """Mock dist.all_gather — every rank's gather_list is populated."""
            rank = self.tls.rank
            self.barrier.wait()
            with self.lock:
                self.scratch.setdefault('agth', [None] * self.world_size)
                self.scratch['agth'][rank] = tensor.detach().clone()
            self.barrier.wait()
            bag = self.scratch['agth']
            for i, src_tensor in enumerate(bag):
                gather_list[i].copy_(src_tensor)
            self.barrier.wait()
            if rank == 0:
                self.scratch.pop('agth', None)
            self.barrier.wait()

        def barrier_op(self):
            self.barrier.wait()


    def _run_fake_world(worker_fn, world_size, *extra_args, timeout=30):
        world = _FakeWorld(world_size)
        errors = [None] * world_size

        def _runner(rank):
            world.tls.rank = rank
            fake_dist = _types.SimpleNamespace()
            fake_dist.ReduceOp = _FakeReduceOp
            fake_dist.all_reduce = lambda tensor, op='SUM': world.all_reduce(tensor, op)
            fake_dist.reduce = lambda tensor, dst, op='SUM': world.reduce(tensor, dst, op)
            fake_dist.broadcast = lambda tensor, src: world.broadcast(tensor, src)
            fake_dist.gather = lambda tensor, gather_list, dst: world.gather(tensor, gather_list, dst)
            fake_dist.all_gather = lambda gather_list, tensor: world.all_gather(gather_list, tensor)
            fake_dist.barrier = world.barrier_op
            fake_dist.get_rank = lambda: rank
            fake_dist.get_world_size = lambda: world_size
            fake_dist.init_process_group = lambda **kw: None
            fake_dist.destroy_process_group = lambda: None
            try:
                worker_fn(rank, world_size, fake_dist, world, *extra_args)
            except BaseException as e:
                import traceback as _tb
                errors[rank] = (e, _tb.format_exc())

        threads = [threading.Thread(target=_runner, args=(r,), daemon=True) for r in range(world_size)]
        for th in threads:
            th.start()
        for th in threads:
            th.join(timeout=timeout)
        for r, err in enumerate(errors):
            if err is not None:
                raise RuntimeError(f'rank {r} failed: {err[0]!r}\n{err[1]}')
        return world.results


    # Odd world_size = 5, values [1, 5, 2, 9, 3] → sorted [1,2,3,5,9] → median 3.
    _vals5 = [1.0, 5.0, 2.0, 9.0, 3.0]

    def _worker5(rank, world_size, dist_module, world):
        world.results[rank] = ex2_all_gather_median(rank, world_size, dist_module, _vals5[rank])

    results = _run_fake_world(_worker5, 5)
    for rank, r in enumerate(results):
        assert r is not None, f'rank {rank} returned None'
        assert abs(r - 3.0) < 1e-5, f'rank {rank}: got {r}, expected 3.0'

    # Even world_size = 4, values [1, 2, 8, 4] → sorted [1,2,4,8] → median (2+4)/2 = 3.
    _vals4 = [1.0, 2.0, 8.0, 4.0]

    def _worker4(rank, world_size, dist_module, world):
        world.results[rank] = ex2_all_gather_median(rank, world_size, dist_module, _vals4[rank])

    results4 = _run_fake_world(_worker4, 4)
    for rank, r in enumerate(results4):
        assert abs(r - 3.0) < 1e-5, f'even rank {rank}: got {r}, expected 3.0'

    # Negative + duplicate values — median must still be stable.
    _vals_neg = [-5.0, -1.0, 0.0, -1.0, 7.0]   # sorted [-5,-1,-1,0,7] → median -1

    def _worker_neg(rank, world_size, dist_module, world):
        world.results[rank] = ex2_all_gather_median(rank, world_size, dist_module, _vals_neg[rank])

    results_neg = _run_fake_world(_worker_neg, 5)
    for rank, r in enumerate(results_neg):
        assert abs(r - (-1.0)) < 1e-5, f'neg/dup rank {rank}: got {r}, expected -1.0'

    # Identical values — median is that value.
    def _worker_same(rank, world_size, dist_module, world):
        world.results[rank] = ex2_all_gather_median(rank, world_size, dist_module, 7.5)

    results_same = _run_fake_world(_worker_same, 3)
    for r in results_same:
        assert abs(r - 7.5) < 1e-5

    # Single-rank degenerate — median of one value is itself.
    def _worker1(rank, world_size, dist_module, world):
        world.results[rank] = ex2_all_gather_median(rank, world_size, dist_module, 42.0)

    results1 = _run_fake_world(_worker1, 1)
    assert abs(results1[0] - 42.0) < 1e-5
    _dd_passed.add('ex2')
    print("ex2 ✓")

_test_ex2()

<details><summary>Solution</summary>

```python
def ex2_all_gather_median(rank: int, world_size: int, dist_module, local_value: float) -> float:
    tensor = t.tensor([local_value], dtype=t.float32)
    gather_list = [t.zeros(1, dtype=t.float32) for _ in range(world_size)]
    dist_module.all_gather(gather_list, tensor)
    gathered = t.cat(gather_list)
    sorted_vals = t.sort(gathered).values
    mid = world_size // 2
    if world_size % 2 == 1:
        return sorted_vals[mid].item()
    return ((sorted_vals[mid - 1] + sorted_vals[mid]) / 2).item()
```

**The list-building step is the easiest place to introduce a bug.** `[t.zeros(1)] * world_size` creates `world_size` references to the SAME tensor — all `world_size` 'slots' alias and the gather silently overwrites itself. Always use a list comprehension to allocate distinct tensors.

**Median vs `kthvalue`.** `t.median` on an even-length tensor returns the LOWER of the two middle values, not their average. Above we compute the average explicitly so the answer matches numpy/scipy conventions. If you actually want torch's lower-mid behavior, `t.median(gathered).values` works for both odd and even.

**`gather` (single dst) vs `all_gather` (everyone).** Use `gather` when only rank 0 needs the result (then broadcast). Use `all_gather` when every rank needs to act on the per-rank values. For a metric you only log on rank 0, gather is cheaper.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()